In [1]:
# @title Install and Import Dependencies

# this assumes that you have a relevant version of PyTorch installed
!pip install -q torchaudio

SAMPLING_RATE = 16000

import os

import torch
import torchaudio
import torchaudio.functional as F

torch.set_num_threads(1)

from pprint import pprint

from IPython.display import Audio

os.chdir("/home/gpu/fuzzy")

pprint(os.getcwd())

/home/gpu/.cache/pypoetry/virtualenvs/traffic-congestion-No1flJwg-py3.9/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'/home/gpu/fuzzy'


In [2]:
# kaiser_fast downsampling
audio, original_sampling_rate = torchaudio.load("95.6_2022_11_29_100000_103000.wav")
resampled_audio = F.resample(
    audio,
    original_sampling_rate,
    SAMPLING_RATE,
    lowpass_filter_width=16,
    rolloff=0.85,
    resampling_method="kaiser_window",
    beta=8.555504641634386,
)

In [3]:
USE_ONNX = False  # change this to True if you want to test onnx model
if USE_ONNX:
    !pip install -q onnxruntime

model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    force_reload=True,
    onnx=USE_ONNX,
)

(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /home/gpu/.cache/torch/hub/master.zip


In [4]:
wav = read_audio("95.6_2022_11_29_100000_103000.wav", sampling_rate=SAMPLING_RATE)
# get speech timestamps from full audio file
speech_timestamps = get_speech_timestamps(wav, model, sampling_rate=SAMPLING_RATE)
pprint(speech_timestamps)

/home/gpu/.cache/pypoetry/virtualenvs/traffic-congestion-No1flJwg-py3.9/lib/python3.9/site-packages/torch/nn/modules/module.py:1190: UserWarning: operator() profile_node %668 : int[] = prim::profile_ivalue(%666)
 does not have profile information (Triggered internally at ../torch/csrc/jit/codegen/cuda/graph_fuser.cpp:105.)
  return forward_call(*input, **kwargs)


[{'end': 314848, 'start': 544},
 {'end': 365024, 'start': 318496},
 {'end': 444384, 'start': 390176},
 {'end': 488416, 'start': 473120},
 {'end': 640480, 'start': 513568},
 {'end': 669664, 'start': 651296},
 {'end': 1131488, 'start': 723488},
 {'end': 2157536, 'start': 1261088},
 {'end': 4208096, 'start': 2169376},
 {'end': 5286368, 'start': 4210208},
 {'end': 5880800, 'start': 5288992},
 {'end': 6498784, 'start': 6015520},
 {'end': 11715040, 'start': 6593568},
 {'end': 11732960, 'start': 11720224},
 {'end': 11772384, 'start': 11751968},
 {'end': 13650400, 'start': 11863072},
 {'end': 14719968, 'start': 13747232},
 {'end': 14741984, 'start': 14736928},
 {'end': 14838240, 'start': 14758944},
 {'end': 16212960, 'start': 14888992},
 {'end': 16249824, 'start': 16219168},
 {'end': 16326112, 'start': 16258592},
 {'end': 16367072, 'start': 16328736},
 {'end': 16418272, 'start': 16372256},
 {'end': 16452064, 'start': 16420896},
 {'end': 16482272, 'start': 16461856},
 {'end': 16553440, 'start':

In [ ]:
# merge all speech chunks to one audio
save_audio(
    "only_speech.wav",
    collect_chunks(speech_timestamps, wav),
    sampling_rate=SAMPLING_RATE,
)
# Audio("only_speech.wav")